In [30]:
"""
Script tự động gán nhãn cho các câu query pháp luật
bằng cách tra cứu trên thuvienphapluat.vn
"""

import json
import time
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# ============================================
# CẤU HÌNH
# ============================================
JSON_INPUT_PATH = "tvpl_congvan3days.json"  # File JSON chứa dữ liệu gốc
QUERIES_PATH = "queries.txt"  # File chứa 100 câu query (mỗi dòng 1 query)
OUTPUT_PATH = "labeled_queries.json"  # File kết quả
MAX_RELEVANT_DOCS = 10  # Số văn bản tối đa cho mỗi query
SEARCH_DELAY = 2  # Thời gian chờ giữa các lần tìm kiếm (giây)


In [31]:

# ============================================
# BƯỚC 1: ĐỌC DỮ LIỆU GỐC
# ============================================
def load_existing_lawids(json_path):
    """Đọc file JSON và lấy tất cả lawid vào set"""
    print(f"📂 Đang đọc file {json_path}...")
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Lấy tất cả lawid vào set
        lawids = set()
        for item in data:
            if 'lawid' in item:
                lawids.add(str(item['lawid']))  # Convert sang string để so sánh
        
        print(f"✅ Đã load {len(lawids)} lawid từ dữ liệu gốc")
        return lawids
    except Exception as e:
        print(f"❌ Lỗi khi đọc file JSON: {e}")
        return set()


In [32]:

# ============================================
# BƯỚC 2: ĐỌC DANH SÁCH QUERIES
# ============================================
def load_queries(queries_path):
    """Đọc file chứa các câu query"""
    print(f"📂 Đang đọc file queries từ {queries_path}...")
    try:
        with open(queries_path, 'r', encoding='utf-8') as f:
            queries = [line.strip() for line in f if line.strip()]
        print(f"✅ Đã load {len(queries)} câu query")
        return queries
    except Exception as e:
        print(f"❌ Lỗi khi đọc file queries: {e}")
        return []


In [33]:

# ============================================
# BƯỚC 3: KHỞI TẠO SELENIUM DRIVER
# ============================================
def init_driver():
    """Khởi tạo Chrome WebDriver"""
    print("🚀 Đang khởi động Chrome WebDriver...")
    options = webdriver.ChromeOptions()
    # options.add_argument('--headless')  # Bỏ comment nếu muốn chạy ngầm
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()
    print("✅ WebDriver đã sẵn sàng")
    return driver


In [34]:
def extract_lawid_from_element(element):
    """Trích xuất lawid từ thuộc tính lawid trong thẻ <p>"""
    try:
        # Tìm thẻ <p> có thuộc tính lawid
        lawid = element.get_attribute('lawid')
        if lawid:
            return lawid
    except:
        pass
    return None

def search_and_scrape(driver, query, existing_lawids):
    """
    Tìm kiếm query trên thuvienphapluat.vn và cào kết quả
    Trả về danh sách các văn bản có lawid trùng với dữ liệu gốc
    """
    relevant_docs = []
    current_page = 1
    
    try:
        # Mở trang chủ
        print(f"  🌐 Đang truy cập thuvienphapluat.vn...")
        driver.get("https://thuvienphapluat.vn/")
        
        # Chờ ô tìm kiếm xuất hiện
        wait = WebDriverWait(driver, 10)
        search_box = wait.until(
            EC.presence_of_element_located((By.XPATH, "//input[@id='txtKeyWord']"))
        )
        
        # Xóa nội dung cũ và nhập query
        print(f"  ⌨️  Đang nhập query...")
        search_box.clear()
        search_box.send_keys(query)
        search_box.send_keys(Keys.RETURN)
        
        # Chờ kết quả tải
        time.sleep(3)
        
        # Lặp qua các trang kết quả
        while len(relevant_docs) < MAX_RELEVANT_DOCS:
            print(f"    📄 Đang cào trang {current_page}...")
            
            # Chờ danh sách kết quả xuất hiện
            try:
                wait.until(
                    EC.presence_of_element_located((By.XPATH, "//p[@class='nqTitle']"))
                )
            except TimeoutException:
                print(f"    ⚠️  Không tìm thấy kết quả trên trang {current_page}")
                break
            
            # Lấy tất cả thẻ <p class="nqTitle"> chứa lawid
            result_elements = driver.find_elements(By.XPATH, "//p[@class='nqTitle']")
            
            if not result_elements:
                print(f"    ⚠️  Không có kết quả nào trên trang {current_page}")
                break
            
            # Duyệt qua từng kết quả
            for idx, element in enumerate(result_elements):
                if len(relevant_docs) >= MAX_RELEVANT_DOCS:
                    break
                
                try:
                    # Lấy lawid từ thuộc tính
                    lawid = extract_lawid_from_element(element)
                    
                    if not lawid:
                        print(f"      ⚠️  Không tìm thấy lawid trong element {idx}")
                        continue
                    
                    # Lấy title từ link bên trong thẻ <a>
                    try:
                        link = element.find_element(By.XPATH, ".//a")
                        # Lấy toàn bộ text bên trong thẻ <a>, bao gồm cả <em>
                        title = link.get_attribute('textContent').strip()
                        
                        # Làm sạch title: loại bỏ khoảng trắng thừa
                        title = ' '.join(title.split())
                    except:
                        print(f"      ⚠️  Không tìm thấy title cho lawid={lawid}")
                        continue
                    
                    # Kiểm tra xem lawid có trong dữ liệu gốc không
                    if lawid in existing_lawids:
                        # Kiểm tra trùng lặp
                        if not any(doc['lawid'] == lawid for doc in relevant_docs):
                            relevant_docs.append({
                                'lawid': lawid,
                                'title': title
                            })
                            print(f"      ✅ Tìm thấy [{len(relevant_docs)}/{MAX_RELEVANT_DOCS}]: lawid={lawid}")
                    else:
                        print(f"      ❌ lawid={lawid} không có trong dữ liệu gốc")
                
                except Exception as e:
                    print(f"      ⚠️  Lỗi khi xử lý kết quả {idx}: {e}")
                    continue
            
            # Nếu đã đủ 10, dừng lại
            if len(relevant_docs) >= MAX_RELEVANT_DOCS:
                break
            
            # Chuyển sang trang tiếp theo
            try:
                # Tìm nút "Trang sau" theo text
                next_button = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable(
                        (By.XPATH, "//a[contains(text(), 'Trang sau') and contains(@href, 'page=')]")
                    )
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
                time.sleep(0.5)
                next_button.click()
                current_page += 1
                time.sleep(2)  # Chờ trang mới tải
            except TimeoutException:
                print(f"    ⚠️  Không tìm thấy nút 'Trang sau' — có thể đã ở trang cuối.")
                break
            except Exception as e:
                print(f"    ⚠️  Lỗi khi click nút Trang sau: {e}")
                break
    
    except TimeoutException:
        print(f"  ❌ Timeout khi tìm kiếm query")
    except Exception as e:
        print(f"  ❌ Lỗi không mong đợi: {e}")
    
    return relevant_docs


In [35]:
def process_all_queries(queries, existing_lawids):
    """Xử lý tất cả các query và thu thập kết quả"""
    driver = init_driver()
    results = []
    failed_queries = []
    
    try:
        for idx, query in enumerate(queries, 1):
            print(f"\n{'='*60}")
            print(f"🔍 [{idx}/{len(queries)}] Đang xử lý query: '{query}'")
            print(f"{'='*60}")
            
            try:
                relevant_docs = search_and_scrape(driver, query, existing_lawids)
                
                results.append({
                    'query': query,
                    'relevant': relevant_docs
                })
                
                print(f"✅ Hoàn thành query '{query}': Tìm được {len(relevant_docs)} văn bản")
                
                # Chờ một chút trước khi query tiếp
                time.sleep(SEARCH_DELAY)
                
            except Exception as e:
                print(f"❌ Lỗi khi xử lý query '{query}': {e}")
                failed_queries.append(query)
                results.append({
                    'query': query,
                    'relevant': [],
                    'error': str(e)
                })
    
    finally:
        driver.quit()
        print("\n🔚 Đã đóng WebDriver")
    
    return results, failed_queries

In [36]:

# ============================================
# BƯỚC 6: LƯU KẾT QUẢ
# ============================================
def save_results(results, output_path):
    """Lưu kết quả ra file JSON"""
    print(f"\n💾 Đang lưu kết quả vào {output_path}...")
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"✅ Đã lưu kết quả thành công!")
    except Exception as e:
        print(f"❌ Lỗi khi lưu file: {e}")


In [38]:
def main():
    print("="*60)
    print("🏛️  AUTO LABELING LEGAL QUERIES")
    print("="*60)
    
    # Bước 1: Load dữ liệu gốc
    existing_lawids = load_existing_lawids(JSON_INPUT_PATH)
    if not existing_lawids:
        print("❌ Không có dữ liệu lawid. Dừng chương trình.")
        return
    
    # Bước 2: Load queries
    queries = load_queries(QUERIES_PATH)
    if not queries:
        print("❌ Không có query nào. Dừng chương trình.")
        return
    
    # Bước 3-4: Xử lý tất cả queries
    results, failed_queries = process_all_queries(queries, existing_lawids)
    
    # Bước 5: Lưu kết quả
    save_results(results, OUTPUT_PATH)
    
    # Thống kê
    print("\n" + "="*60)
    print("📊 THỐNG KÊ KẾT QUẢ")
    print("="*60)
    print(f"Tổng số query: {len(queries)}")
    print(f"Query thành công: {len(queries) - len(failed_queries)}")
    print(f"Query thất bại: {len(failed_queries)}")
    
    if failed_queries:
        print(f"\n❌ Các query bị lỗi:")
        for q in failed_queries:
            print(f"  - {q}")
    
    total_docs = sum(len(r['relevant']) for r in results)
    avg_docs = total_docs / len(results) if results else 0
    print(f"\nTổng văn bản tìm được: {total_docs}")
    print(f"Trung bình mỗi query: {avg_docs:.2f} văn bản")
    print("="*60)

if __name__ == "__main__":
    main()

🏛️  AUTO LABELING LEGAL QUERIES
📂 Đang đọc file tvpl_congvan3days.json...
✅ Đã load 30630 lawid từ dữ liệu gốc
📂 Đang đọc file queries từ queries.txt...
✅ Đã load 100 câu query
🚀 Đang khởi động Chrome WebDriver...
✅ WebDriver đã sẵn sàng

🔍 [1/100] Đang xử lý query: 'thuế thu nhập cá nhân là gì'
  🌐 Đang truy cập thuvienphapluat.vn...
  ⌨️  Đang nhập query...
    📄 Đang cào trang 1...
      ❌ lawid=581475 không có trong dữ liệu gốc
      ❌ lawid=557149 không có trong dữ liệu gốc
      ❌ lawid=533850 không có trong dữ liệu gốc
      ❌ lawid=499280 không có trong dữ liệu gốc
      ❌ lawid=467715 không có trong dữ liệu gốc
      ❌ lawid=435698 không có trong dữ liệu gốc
      ❌ lawid=423854 không có trong dữ liệu gốc
      ❌ lawid=370829 không có trong dữ liệu gốc
      ❌ lawid=326627 không có trong dữ liệu gốc
      ❌ lawid=310639 không có trong dữ liệu gốc
      ❌ lawid=513425 không có trong dữ liệu gốc
      ❌ lawid=445345 không có trong dữ liệu gốc
      ❌ lawid=567714 không có trong 